TASK 2: Syntax Familiarisation (Line-Matching)
   Before we write `intvec_init`, let's ... C syntax parsing. Match the 
   C-snippets (1-6) to their practical effects (A-F).


1. `size_t`
   An unsigned integer type guaranteed to be large enough to contain the size
   of the biggest object the host system can handle.
2. `v->cap`
   Dereferences a struct pointer to acess a specific member vairable. 
3. `malloc(new_cap * sizeof *v->data)`
   Allocates a raw block of memory safely, automatically scaling by the 
   byte-size of the underlying type rather than hardcoding `sizeof(int)`.
4. `free(v->data)`
   Releases a previosuly allocated heap block back to the system, preventing
   memory leaks.
5. `int *out`
   An output parameter (pointer to the caller's memory) used to "return" data
   when the actual function return value is used for error codes. 
6. `return -1;`
   Standard C idiom to indicate an error occured (e.g., allocation failed or 
   index out of bounds).


   - Think of `data` as the raw buffer, and the struct itself as the "manager"
     of that buffer.
   - Output parameters (lik `int *out`) are crucial in C because a function can
     only return one value. If you return `0` or `-1` for success/failure, you
     need another way to hand the actual data back to the caller. 

SYNTAX HINTS:
   - `typedef struct { ... } Name;` creates an anonymous struct and aliases it 
     to `Name`, saving you from typing `struct IntVec` everywhere.
   - `sizeof *pointer` is highly preferred over `sizeof(Type)`. If the type of
     the pointer ever cahnges during refactoring, `sizeof *pointer` automatically
     adapts. `sizeof(Type)` does not, which leads to silent buffer overflows.

     

---

THE BASE STANDARD
   The `-std=c17` forces the compiler to use the ISO C17 standard to parse your 
   code. This ensures you are using modern, standardised C rules rather than
   obsolete 1989 synjtax or non-standard compiler extensions. It turns off 
   GNU-specific dialects by default, which ensures that the code you write on
   MacOS or Linux remains highly portable across different platforms, compilers
   or hardware targets.

THE STRICT WARNING SUITE
   The `-Wall` and `-Wextra` flags act as your first line of defense against 
   logic bugs. Despite its name, "Wall" does not enable all warnings, but rather
   a core set of reliable checks for high-risk mistakesw like uninitialised 
   variables of missing return statements. Adding `-Wextra` layers on even 
   stricter chedcks,  flagging things like comparing signed integers with 
   unsigned integers, which frequently causes silent bugs in loop conditions.

   To keep your code strictly disciplined, `-Wpedantic` enforces total compliance
   with the ISO C17 standard. It will trigger a warning for any code that 
   relies on compiler-specific extensions or forbidden behaviors, ensuring your
   code remains entirely clean. Combined with the other warning flags
   (`-Wall -Wextra` ++ `-Wpedantic`)... it forces you to write code that passes
   the strictest scrutiny before it even attempts to compile.

DEFENSIVE TYPE AND SCOPE CHECKS
   The `-Wconversion` fag is critical when dealing with memory allocation and
   array indexing. It warns you anytime data might be implicitly hidden or 
   altered during a type conversion--such as assigning a negative `int` to an
   unsigned `size_t` variable, or downcasting a 64-bit integer into a 32-bit 
   integer. This forces you to write explciit typecasts, preventing silent
   truncationb bugs that frequently lead to memory corruption. 

   The `-Wshadow` flag protects the integrity of your variable names by warning
   if a local variable shares a name with a variable in an outer scope. For
   instance, if you decalre a loop counter named `i` inside a block that already
   has access to a function parameter named `i`, the compiler will flag it. This
   stops you from accidentally modifying or reading the wrong variable due to an 
   accidental name collision.

DEBUGGING AND INSTRUMENTATION
   The `-g` flag instructs the compiler to generate comprehensive debug symbols
   and embed them directly inside the compiled binary. This maps the generated 
   machine code instruction back to the exact file names and line numbers of 
   your C source code. Without `-g`, tools like `lldb`, `gdb`, or crash dump
   analyzers can only show you raw assembly addresses rather than the 
   human-readable code you actually wrote. 

   The `-fsanitze=address,undefined` flag injects active instrumentation hooks
   directly into your executable to catch runtime errors. The AddressSanitizer
   (`asan`) sets up "poison zones" around your allocations to instantly crash
   and report memory bugs like bugger overflows, use-after-free errors, or 
   double frees. The UndefinedBehaviorSanitizer (`ubsan`) simultaneously 
   tracks down runtime illegalities, such as integer overflow or null pointer 
   dereferences.

   Finally, `-fno-omit-frame-pointer` forces the compiler to dedicate a hardware
   register to tracking the function call stack frame. Optimisation passes often
   discard this register to squeeze out peerformance, but keeping it ensures
   your debugging tools and sanitizers can accurately reconstruct the exact
   execution path leading up to a crash. It guarantees that when a sanitizer
   catches a memory leak, the printed stack trace is peerfectly accurate and 
   complete.


   sau nofp
   AEPSCG
   `-fsanitize=address,undefined -fno-omit-frame-pointer`
   `cmake -std=c17 -Wall -Wextra -Wpedantic -Wshadow -Wconversion -g`


   












   `-fsanitize=address,undefined -fno-omit-frame-pointer`
   `-fsanitize=address,undefined -fno-omit-frame-pointer`

    AEPCSG -- -Wall -Wextra -Wpedantic -Wconversion -Wshadow -g
   `cmake -std=c17 -Wall -Wextra -Wpedantic -Wconversion -Wshadow -g`


   `-fsanitize=address,undefined`

---
   INSTRUMENTATION HOOKS in C are special function or mechanism used to monitor,
   trace, or modify a program's execution at runtime without altering its 
   original source code. They are primarily used for debugging, profiling
   (measuring performance), code coverage analysis, and security auditing.

   1. Compiler-Generated Hooks (GCC/Clang)
      Modern compilers provide built-in flags that automatically inject entry
      and exit hooks into every function.
      - THE FLAG: Compiling with `-finstrument-functions` tells the compiler to
        insert a call to `__cyg_profile_func_enter` at the start of a function
        and  `__cyg_profile_func_exit` at the end.
      - USE CASE: You write the custom logic for these two functions. This is 
        heavily utilised in profilers (like `gprof`) to buil;d execution trees
        and measure time spent in different parts of the code.   
      - EXAMPLE:

```c
#include <stdio.h>

// __attribute__((no_instrument_function)) is vital here
// to prevent an infinite loop of hooks calling hooks!
__attribute__((no_instrument_function))
void __cyg_profile_func_enter(void *this_fn, void *call_site) {
    printf("Entering function at address %p\n", this_fn);
}

__attribute__((no_instrument_function))
void __cyg__profile_func_exit(void *this_fn, void *call_site) {
    printf("Exiting function at address %p\n", this_fn);
}

void main_logic() {
    printf("Doing work...\n");
}

int main() {
    main_logic();
    return 0;
}
```

...
   ...
...


COMMON USE CASES AT A GLANCE
   - PROFILING: Measuring the exact amount of CPU time a function takes to run.
   - CODE COVERAGE: Seeing which functions were actually executed during a 
     testing phase.
   - TRACING/LOGGING: Printing a log trace whenever an event occurs without
     littering the core buisness logic with `printf` statements. 

   ... `free()` is exactly the functional opposite of `malloc()`. While 
   `malloc()` requests the operating system to reserve a contiguous block of 
   memory on the heap and return a pointer to it, `free(ptr)` tells the system's
   memory manager that you are done using that specific block, allowing that
   address space toi be recycled and reused for future allocations... 
   Crucially, `free(0)` only relinquishes ownership of the memory back to the 
   system; it does not clear the data inside the block, nor does it modify the 
   pointer variable itself, leaving it as a dangerous "dangling pointer" that
   still points to tbhe now-deallocated address until you manually set it to
   `NULL`.

THE SYNTAX RULE
   In C, `sizeof` is an operator, not a function.
   - If you give `sizeof` a TYPE NAME, parentheses are MANDATORY : `sizeof(int)`
   - If you give `sizeof` an EXPRESSION/VARIABLE, parentheses are OPTIONAL:
     `sizeof *v->data` or `sizeof(*v->data)`.

   Both compile to the exact same machine code, but writing it without 
   parentheses is a common idiom among experienced systems eninggers. ... why
   it works and why people do it...

---
PRECEDENCE: Why `sizeof *v->data` WORKS WITHOUT PARENS
   The arrow operator `->` has higher precedence than the unary dereference 
   operator `*`.

   When the compiler evaluates `sizeof *v->data`, it parses it in this order:
   1. `v->data` (Access the `data` pointer inside the struct `v`)
   2. `*` (Deereference that pointer to get the actual `int` value it points to).
   3. `sizeof` (Evaluate the size of that `int`)

   Because `->` happens first... don't need parentheses to force the order of 
   the operations. 

---
Why `sizeof *ptr` is preferred over `sizeof(Type)`
   The real reason to write `sizeof *v->data` (with or without parens) instead
   of `sizeof(int)` is TYPE SAFETY DURING FUTURE REFACTORING.

   Imagine you later change the dynamic array to hold a larger type, like
   `long long` or a custom `struct SensorReading`:

```c
typedef struct {
    long long *data;
    size_t len;
    size_t cap;
} IntVec;
```
   - THE RISKY WAY: If your allocation line was `malloc(new_cap * sizeof(int))`,
     your code will now compile without warnings, but you will allocate HALF the
     memory you actually need. This leads to silent buffer ovierflows and 
     brutal debugging sessions.
   - THE SAFE WAY: If your allocation line is 
     `malloc(new_cap * sizeof *v->data)`... compiler automatically updates the
     size evaluation to `sizeof(long long)` because it looks directly at the 
     type of the pointer. Your allocation remains perfectly correct without
     changing a line of implementation code...

   Whether you prefer `sizeof *v->data`... stylistic choice.... The 
   un-parenthesized veersion is just a common C habit to reduce visual noise...  

   ... The arrow operator `->` is a syntactic shortcut that combines two distinct
   steps: it first DEREFERENCES a pointer to a struct, and then it ACCESSES a
   specific field within that struct. Writing `v->len` is completely identical
   to writing `(*v).len`. The explicit parentheses in `(*v).len` are mandatory
   because the dot operator `.` has higher precedence than the dereference
   operator `*`; without them, `*v.len` would mistakenly try to dereference a 
   field named `len` inside a non-pointer variable `v`. The arrow operator 
   ... desgined to eliminate this clunky syntax, providing a clean, un-parenthesized
   way to traverse pointers to structures. 

---

sau-nofp
AEPSCG
`-fsanitze=address,undefiend -fno-omit-frame-pointer`
`cmake -std=c17 -Wall -Wextra -Wpedantic -Wshadow -Wconversion -g`

---

---

sau nofp
aescpg
`-fsanitize=address,undefined -fno-omit-frame-pointer`
`cmake -std=c17 -Wall -Wextra -Wpedantic -Wshadow -Wconversion -g`

   - Compile cleanly before optimising
   - Run with sanitisers before trusting memory code.
   - Keep generated build files out of Git.
   - Prefer reproductible commands over IDE-only build.

---
   - Write small code
   - Compile with strict warnings
   - Run tests
   - Run sanitizers
   - Debug failures
   - Repeat

   ... `cmake` is not the compiler. `clang` or `gcc` is the compiler. `cmake`
   is a build-system generator that can produce Makefiles/Ninja projects.
   So this is ...

---
   ... maths-like aspect: C quality is largely about maintaining invariants. For
   `IntVec`, the core invariant is: 

```
0 <= len <= cap
data != NULL iff cap >0
data[0..len) contains valid elements
data[len..cap) is allocated but logically unused
```

   ... breakdown of exactly how these mechanisms work in C.

C ENUMS vs. KOTLIN ENUM CLASSES
   In C, an `enum` is essentially just a list of glorified `int` constants. 
   There is ZERO TYPE SAFETY and NO NAMESPACES.
   - Kotlin's `enum class`: In Kotlin, enums are strongly typed objects. You
     access them via their namespace (e.g., `Status.OK`), and they can have
     properties, methods, and strict type checking.
   - C's `enum`: C dumps all the enum values directly into the global (or file)
     scope. At compile time, they just become raw integers.

   This lack of scoping ois eaxctly why C developers use heav prefixing like
   `INTVEC_OK` or `INTVEC_ERR_ALLOC`. If you just named the value `OK`, it would
   fatally collide with any other enum or variable named `OK` anywhere else in#
   your codebase.

   Because C treats them as raw integers, you can easily return them from 
   functions like `int intvec_push()` and evaluate them directly.

HEADER GUARDS (`#ifndef` / `#define` / `#endif`)
   ... but the preprocessor doesn't check if the macro is "true"--it checks if
   the macro EXISTS.

   `#ifndef` stands for "If Not Defined". ... what happens during compilation:
   1. The very first time a C file includes `intvec.h`, the preprocessor checks
      if the name `PROJECT_C_INTVEC_H` exists in its internal directory.
   2. doesn't exist... enter the block...
   3. very next line... `#define PROJECT_C_INTVEC_H`... immediately adds that
      name to the dictionary...
   4. The preprocessor then reads and processes all your structs, enums, and#
      function prototypes up until the `#endif`.

WHY DO WE DO THIS?
   In C, you often have multiple files including the same header. If `file_a.c`      
   includes `intvec.h`... the compileer will accidentally try to paste the 
   contents of `intvec.h` twice. 

   Without those header guards, the compiler would see 
   `typedef struct { ... } IntVec;` two times in a row and crash with a 
   "redefinition error." Because of the `#ifndef` check, the second time the
   preprocessor seeds the file, it realises... already defined, and safely
   ignores everything inside the block. 


---

CONCEPT: Forward Declarations and "Incomplete Types"
   C compilers read files strictly top-to-bottom. If `controller.h` needs to
   store a pointer to a `MotorState`, but the actual struct is defined in 
   `motor.h`, the compileer will panic when it sees an unknown name.

   A forward declaration tells the compileer: "Trust me, this struct exists 
   somewhere else. Treat it as an `incomplete type` for now."

   Because the type is incomplete, the compiler doesn't know how big it is.
   Therefore, you CANNOT declare it by value (`MotorState m;`) or access its
   fields (`m->speed`). You CAN ONLY USE POINTERS (`MotorState *m`); because
   the compiler already knows exactly how big a pointer is (e.g., 8 bytes on
   64-bit system).

   ... exact difference between your two lines...

---
1. `struct MotorState;`
   This is the raw C forward declaration.
   - What it does: It tells the compiler that a `struct` named `MotorState`
     exists.
   - HOW YOU USE IT: ... forced to inlcude the `struct` keyword eveery time you
     reference it. 
   - WHEN TO USE IT: When you are working in a codebase that strictly forbids
     typedefs for structs (like the Linux Kernel), or if you just need to break 
     a quick circular dependency inside a single `.c` file.

2. `typedef struct MotorState MotorState;`
   This is a forward declaration combined with a type alias.
   - What it does: It tells the compiler the struct exists, AND immediately 
     aliases it so you never have to type `struct` again.
   - HOW YOU USE IT:

```c
typedef struct MotorState MotorState;

void update_motor(MotorState *m);
```
   - WHEN TO USE IT: ... what to normally use...

THE OPAQUE POINTER PATTERN (Why this matters)
   If you put `typedef struct MotorState MotorState;` in your header file 
   (`motor.h`), but you hide the actual strict definition 
   (`struct MotorState { int temp; ...}}`) inside your source file (`motor.c`),
   you achieve true encapsulation.

   Other parts of your program can pass `MotorState *` pointers around, but they
   PHYSICALLY CANNOT mess with the struct's internal fields because the compiler
   doesn't know what they are. They are forced to use your API functions, like
   `motor_get_temp(MotorState *m)`.


3. Task 3: Implementing `intvec_init` && `intvec_free`
   now that your syntax parser is warmped u p, let's ...

```c
// invec.h
typedef struct {
    int *data;
    size_t len;
    size_t cap;
} IntVec;

int intvec_init(IntVec *v);
void intvec_free(IntVec *v);
```

CONCEPTUAL HINTS:
   * `init`: A new vector should start with a small capacity (e.g., 16), a length
     of 0, and a fresh heap allocation for `data`. If the allocation fails, 
     return `-1`. If it succeeds, return `0`.
   * `free`: You must release the heap memory, but it;s also good defensive
     programming to zero out the pointer and lengths so that a "use-after-free"
     bug will crash loudly and immediately rather than corrupting memory 
     silently.

---

   ... caused by a historical quirk in how the C compiler categorises names.

   In C, `typedef` always follows this grammar:
   `tpedef [EXISTING TYPE] [NEW_ALIAS];`

   For example;
   `typedef unsigned long long int uint64_t;`

   When you write `typedef struct MotorState MotorState;` you are doing the exact
   same thing:

   - `[EXINSTING_TYPE]` = `struct MotorState`
   - `[NEW_ALIAS]` = `MotorState`


WHY DO THEY SHARE THE SAME NAME?   
   In C, structs live in a completely separate "namespace" (technically called
   the tag namespace) from normal variables and types.

   Because they live in different namespaces, the compiler does not get confused
   if a struct tag and a type alias share the exact same word.

   You do not have to make them the same. You will frequently see C codebases
   do this... instead to make the difference obvious:

```c
// Using _s to denote the struct tag, and _t to denote the typedef alias
```
   However, modern C developers often find appending `_s` or `_t` everywhere to
   be visual clutter. Because the compileer allows the struct tag and the 
   typedef alias to be identical without conflict, it has become standard 
   practice to just double the name.
   1. The first `MotorState` reserves the name in the STRUCT TAG NAMESPACE.
   2. The second `MotorState` creates an alias in the STANDARD TYPE NAMESPACE.

   ... 


---

   `<stddef.h>` is the "Standard Definitions" header in C. It is ubiquitous 
   because it provides the fundamental vocabular C needs to interact with memory
   safely across different hardware architectures. Rather than containing actual
   executable functions, it defines the core types and macros that make 
   portable C system programming possible. 

   The vast majority of the time, it is included for three specific things:
   `size_t`, `NULL`, and `ptrdiff_t`. As we just convered, `size_t` safely 
   represents the byte-size or length of any object. `NULL` is the universal
   constant for a pointer that points nowhere. `ptrdiff_t` is the signed 
   counterpart to `size_t`; it is the only integer type to use when you 
   subtract one pointer from another to find the distance between them in 
   memory.

   Finally, it contains the `offsetof(type, member)` macro. When writing parsers
   , handling hardware registers, or dealing with binary telemetry packets, you
   often need to know exactly how many byts into a `struct` a specific field 
   lives. `<stddef.h>` gives you this exact byte-offset safely, regardless of how
   the compiler decides to pack or pad the struct in memory...

   - `size_t`, `NULL`, `ptrdiff_t`

---